# Süperiletken Josephson Junction Kuantum Çip — Uçtan Uca Tasarım & Simülasyon

**Quantum Metal (eski Qiskit Metal) + scqubits + QuTiP + Stim/PyMatching**

Bu notebook tek bir transmondan başlayıp, onu çoğaltarak bir **mantıksal qubit** (mesafe-3 tekrarlama kodu) kurar ve hata düzeltmenin çalıştığını simülasyonda gösterir.

| Faz | İçerik |
|---|---|
| 0 | Kurulum + doğrulama |
| 1 | Transmon + okuma rezonatörü (geometri) |
| 2 | LOM → Hamiltonyen parametreleri |
| 3 | Kuantumlama: dispersif okuma, T1/T2, DRAG kapısı |
| 4 | İki-qubit CZ kapısı |
| 5 | Tam çip layout'u (5 qubit) + GDS |
| 6 | Mantıksal qubit: devre-seviyesi hata düzeltme |

> **Önemli:** Hücreleri **sırayla, tek tek** çalıştır. Faz 0a'dan sonra **Runtime > Restart session** yap, sonra 0b'den devam et.

## Faz 0a — Kurulum  ⚠️ Bu hücreden sonra `Runtime > Restart session` yap

In [ ]:
# === FAZ 0a: Kurulum ===
!pip -q install quantum-metal qutip scqubits stim pymatching
# quantum-metal numpy'i <2'ye dusurur ama qutip/scqubits/stim 2.x ister.
# Cozum: numpy'i TEK ve tutarli bir 2.x surume FORCE-REINSTALL ile temiz kur.
# (--upgrade yarim/bozuk birakabiliyor -> '_center' ImportError; force-reinstall onu engeller)
!pip -q install --force-reinstall --no-deps "numpy==2.3.5"
print(">>> SIMDI: ust menu  Runtime > Restart session  -> sonra Faz 0b'den devam et")
print(">>> Faz 0b'de hala numpy import hatasi olursa: bu hucreyi tekrar calistir + tekrar restart.")

## Faz 0b — Doğrulama (restart'tan sonra)

In [ ]:
# === FAZ 0b: Dogrulama ===
import numpy as np
print("numpy:", np.__version__)        # 2.1-2.3 gormeliyiz
import qiskit_metal as qm, qutip, scqubits, stim, pymatching
print("Quantum Metal:", qm.__version__, "| qutip:", qutip.__version__,
      "| scqubits:", scqubits.__version__, "| stim:", stim.__version__)

## Faz 1 — Tek transmon + çeyrek-dalga okuma rezonatörü

In [ ]:
# === FAZ 1 ===
import qiskit_metal as qm
from qiskit_metal import designs, Dict
from qiskit_metal.qlibrary.qubits.transmon_pocket import TransmonPocket
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround

design = designs.DesignPlanar()
design.chips.main.size.size_x = '6mm'
design.chips.main.size.size_y = '4mm'
design.overwrite_enabled = True

q1 = TransmonPocket(design, 'Q1', options=dict(
    pos_x='0mm', pos_y='0mm',
    pad_width='455um', pad_height='90um', pad_gap='30um',
    connection_pads=dict(
        readout=dict(loc_W=+1, loc_H=+1, pad_width='80um'),
        drive  =dict(loc_W=-1, loc_H=+1, pad_width='80um'),
    )))
OpenToGround(design, 'otg', options=dict(pos_x='2.2mm', pos_y='0mm', orientation='180'))
RouteMeander(design, 'readout_res', options=Dict(
    total_length='4.3mm', fillet='90um', hfss_wire_bonds=False,
    pin_inputs=Dict(start_pin=Dict(component='Q1', pin='readout'),
                    end_pin  =Dict(component='otg', pin='open'))))

fig = qm.view(design); fig.savefig('faz1_cip.png', dpi=140)
print("Faz 1 cizimi hazir.")
fig

## Faz 2 — LOM → Hamiltonyen parametreleri

In [ ]:
# === FAZ 2 ===
import numpy as np
e=1.602176634e-19; h=6.62607015e-34; hbar=h/(2*np.pi); Phi0=h/(2*e)

# Bu geometriye uygun temsili lumped degerler (Colab'da Ansys Q3D yerine).
# Ansys varsa: quantum-metal[ansys] + LOManalysis ile matris gercek Q3D'den cikar; akis ayni.
C_sigma, C_g, C_r = 88e-15, 6e-15, 400e-15
Lj, f_r = 10e-9, 7.0e9
E_C = e**2/(2*C_sigma)/h
E_J = (Phi0/(2*np.pi))**2/Lj/h

def transmon(EC, EJ, ng=0.0, ncut=40):
    n=np.arange(-ncut,ncut+1)
    H=np.diag(4*EC*(n-ng)**2)+np.diag(-EJ/2*np.ones(2*ncut),1)+np.diag(-EJ/2*np.ones(2*ncut),-1)
    w,v=np.linalg.eigh(H); i=np.argsort(w); w,v=w[i],v[:,i]
    return w-w[0], abs(v[:,0]@np.diag(n)@v[:,1])

E, n01 = transmon(E_C, E_J)
f01, alpha = E[1]-E[0], (E[2]-E[1])-(E[1]-E[0])
beta=C_g/C_sigma
g_Hz = (2*beta*e*np.sqrt(hbar*2*np.pi*f_r/(2*C_r))*n01/hbar)/(2*np.pi)

params = dict(E_C=E_C, E_J=E_J, f01=f01, alpha=alpha, f_r=f_r, g=g_Hz, Delta=f01-f_r)
for k,v in params.items(): print(f"{k:8}= {v/1e6:9.2f} MHz")
print(f"\nE_J/E_C = {E_J/E_C:.1f} | g/|Delta| = {g_Hz/abs(f01-f_r):.3f} (dispersif rejim)")

## Faz 3a — Kuplajlı spektrum + dispersif okuma (scqubits)

In [ ]:
# === FAZ 3a ===
import numpy as np, matplotlib.pyplot as plt
import scqubits as scq

EJ, EC = params['E_J']/1e9, params['E_C']/1e9
fr, g  = params['f_r']/1e9, params['g']/1e9
tmon = scq.Transmon(EJ=EJ, EC=EC, ng=0.0, ncut=31)
ev = tmon.eigenvals(evals_count=4)
f01 = ev[1]-ev[0]; alpha = tmon.anharmonicity()
print(f"scqubits: f01 = {f01*1e3:.1f} MHz | anharmoniklik = {alpha*1e3:.1f} MHz")

Delta = f01 - fr
chi = g**2 * alpha / (Delta*(Delta+alpha))
print(f"Delta = {Delta*1e3:.0f} MHz | chi/2pi = {chi*1e6:.0f} kHz | 2chi/2pi = {2*chi*1e6:.0f} kHz")

kappa = 1.5e-3
det = np.linspace(-3e-3, 3e-3, 2000)
S = lambda d,s: np.abs(1-(kappa/2)/((kappa/2)+1j*(d-s)))
plt.figure(figsize=(7,4))
plt.plot(det*1e3, S(det,+chi), color="#1f3b73", lw=2.4, label="qubit |0>")
plt.plot(det*1e3, S(det,-chi), color="#c0392b", lw=2.4, label="qubit |1>")
plt.xlabel("f - f_r (MHz)"); plt.ylabel("|S21|")
plt.title(f"Dispersif okuma  (2chi = {abs(2*chi)*1e3:.2f} MHz)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
params.update(chi=chi*1e9, kappa=kappa*1e9)

## Faz 3b-i — T1/T2 koherans (QuTiP master denklemi)

In [ ]:
# === FAZ 3b-i ===
import qutip as qt, numpy as np, matplotlib.pyplot as plt
T1, T2 = 50e3, 40e3
g1, gphi = 1/T1, 1/T2 - 1/(2*T1)
a2 = qt.destroy(2)
c_ops = [np.sqrt(g1)*a2, np.sqrt(2*gphi)*a2.dag()*a2]
tl = np.linspace(0, 200e3, 400)
r1 = qt.mesolve(0*a2, qt.basis(2,1), tl, c_ops, e_ops=[a2.dag()*a2])
det = 2*np.pi*0.5e-3
r2 = qt.mesolve(det*a2.dag()*a2, (qt.basis(2,0)+qt.basis(2,1)).unit(), tl, c_ops, e_ops=[qt.sigmax()])

plt.figure(figsize=(11,4))
plt.subplot(1,2,1); plt.plot(tl/1e3, r1.expect[0], '#1f3b73', lw=2)
plt.plot(tl/1e3, np.exp(-tl/T1), 'k--', lw=1, label='exp(-t/T1)')
plt.xlabel('us'); plt.ylabel('uyarilmis pop.'); plt.title(f'T1={T1/1e3:.0f} us'); plt.legend(); plt.grid(alpha=.3)
plt.subplot(1,2,2); plt.plot(tl/1e3, r2.expect[0], '#c0392b', lw=1)
plt.plot(tl/1e3, np.exp(-tl/T2), 'k--', lw=1, label='exp(-t/T2) zarfi')
plt.xlabel('us'); plt.ylabel('<sx>'); plt.title(f'T2={T2/1e3:.0f} us'); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

## Faz 3b-ii — DRAG X kapısı (3-seviye, sızıntı bastırma)

In [ ]:
# === FAZ 3b-ii ===
import qutip as qt, numpy as np, matplotlib.pyplot as plt
alpha = params['alpha']/1e9
a = qt.destroy(3)
H0 = 2*np.pi*alpha*(a.dag()*a.dag()*a*a)/2
Hx, Hy = 0.5*(a+a.dag()), 0.5j*(a.dag()-a)

def pulse(sig, drag):
    t0, A = 3*sig, np.pi/(sig*np.sqrt(2*np.pi))
    gx = lambda t, **kw: A*np.exp(-(t-t0)**2/(2*sig**2))
    gy = lambda t, **kw: (-A*(-(t-t0)/sig**2)*np.exp(-(t-t0)**2/(2*sig**2))/(2*np.pi*alpha)) if drag else 0.0
    return [H0,[Hx,gx],[Hy,gy]], 6*sig

H, tg = pulse(4.0, True); tl = np.linspace(0,tg,400)
res = qt.sesolve(H, qt.basis(3,0), tl, e_ops=[qt.ket2dm(qt.basis(3,i)) for i in range(3)])
print(f"DRAG X (tg={tg:.0f}ns): P0={res.expect[0][-1]:.4f} P1={res.expect[1][-1]:.4f} sizinti={res.expect[2][-1]:.2e}")

sigs = np.linspace(1,7,20)
def leak(s, d):
    H,_ = pulse(s,d)
    return qt.sesolve(H, qt.basis(3,0), np.linspace(0,6*s,300),
                      e_ops=[qt.ket2dm(qt.basis(3,2))]).expect[0][-1]
ln, ld = [leak(s,False) for s in sigs], [leak(s,True) for s in sigs]
plt.figure(figsize=(7,4))
plt.semilogy(6*sigs, ln, 'o-', color='#c0392b', label='DRAG yok')
plt.semilogy(6*sigs, ld, 's-', color='#1f3b73', label='DRAG var')
plt.xlabel('kapi suresi tg (ns)'); plt.ylabel('|2> sizinti'); plt.title('DRAG sizintiyi bastiriyor')
plt.legend(); plt.grid(alpha=.3, which='both'); plt.tight_layout(); plt.show()

## Faz 4 — İki-qubit CZ kapısı (QuTiP)

In [ ]:
# === FAZ 4 ===
import qutip as qt, numpy as np, matplotlib.pyplot as plt
twopi=2*np.pi; N=3
a=qt.destroy(N); I=qt.qeye(N)
a1,a2 = qt.tensor(a,I), qt.tensor(I,a)
n1,n2 = a1.dag()*a1, a2.dag()*a2
w1,al1,al2 = 5.135,-0.245,-0.245
w2 = w1 - al2; J = 0.012
H = twopi*( w1*n1 + al1/2*n1*(n1-1) + w2*n2 + al2/2*n2*(n2-1) + J*(a1.dag()*a2 + a1*a2.dag()) )

ix = lambda q1,q2: q1*N+q2
ts = np.linspace(0,200,1500)
P11,P02,phi = [],[],[]
for t in ts:
    U = (-1j*H*t).expm().full()
    P11.append(abs(U[ix(1,1),ix(1,1)])**2)
    P02.append(abs(U[ix(0,2),ix(1,1)])**2)
    ph = (np.angle(U[ix(1,1),ix(1,1)])-np.angle(U[ix(0,1),ix(0,1)])
          -np.angle(U[ix(1,0),ix(1,0)])+np.angle(U[ix(0,0),ix(0,0)]))
    phi.append((ph+np.pi)%twopi-np.pi)
P11,P02,phi = map(np.array,(P11,P02,phi))
back = np.where((P11>0.999)&(ts>5))[0]; tcz=ts[back[0]]
print(f"t_CZ = {tcz:.1f} ns | kosullu faz = {phi[back[0]]/np.pi:.3f} pi | sizinti = {1-P11[back[0]]:.1e}")

fig,(ax,bx)=plt.subplots(1,2,figsize=(12,4.5))
ax.plot(ts,P11,'#1f3b73',lw=2,label='P|11>'); ax.plot(ts,P02,'#c0392b',lw=2,label='P|02>')
ax.axvline(tcz,color='gray',ls=':'); ax.set_xlabel('ns'); ax.set_ylabel('pop.'); ax.legend(); ax.grid(alpha=.3); ax.set_title('CZ tam tur')
bx.plot(ts,phi/np.pi,'#1e8449',lw=2); bx.axhline(1,color='gray',ls='--'); bx.axvline(tcz,color='gray',ls=':')
bx.set_xlabel('ns'); bx.set_ylabel('kosullu faz / pi'); bx.grid(alpha=.3); bx.set_title('-> pi')
plt.tight_layout(); plt.show()
gates = dict(f_1q=0.999, f_2q=0.999, t_1q=24, t_2q=float(tcz))

## Faz 5 — Tam çip layout'u (5 qubit) + GDS

GDS export'u bu sürümde geometri renderer'ında pürüzlü olabiliyor; bu yüzden `try/except` içinde — **patlasa bile zincir durmaz**, çip çizimi (asıl çıktı) gene oluşur.

In [ ]:
# === FAZ 5 ===
import qiskit_metal as qm
from qiskit_metal import designs, Dict
from qiskit_metal.qlibrary.qubits.transmon_pocket import TransmonPocket
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround

design = designs.DesignPlanar()
design.chips.main.size.size_x = '13mm'
design.chips.main.size.size_y = '6mm'
design.overwrite_enabled = True

labels = ['D0','A0','D1','A1','D2']; xs = [-4,-2,0,2,4]
for name, x in zip(labels, xs):
    TransmonPocket(design, name, options=dict(
        pos_x=f'{x}mm', pos_y='0mm',
        pad_width='455um', pad_height='90um', pad_gap='30um',
        connection_pads=dict(
            readout=dict(loc_W=+1, loc_H=+1, pad_width='80um'),
            left   =dict(loc_W=-1, loc_H=-1, pad_width='80um'),
            right  =dict(loc_W=+1, loc_H=-1, pad_width='80um'))))
    OpenToGround(design, f'otg_{name}', options=dict(pos_x=f'{x}mm', pos_y='2.4mm', orientation='-90'))
    RouteMeander(design, f'res_{name}', options=Dict(
        total_length='4.3mm', fillet='60um', hfss_wire_bonds=False,
        pin_inputs=Dict(start_pin=Dict(component=name, pin='readout'),
                        end_pin  =Dict(component=f'otg_{name}', pin='open'))))
for n1, n2 in zip(labels[:-1], labels[1:]):
    RouteStraight(design, f'cpl_{n1}_{n2}', options=Dict(
        pin_inputs=Dict(start_pin=Dict(component=n1, pin='right'),
                        end_pin  =Dict(component=n2, pin='left'))))

fig = qm.view(design); fig.savefig('faz5_cip.png', dpi=150)
print("Cip cizimi: 5 qubit + 5 okuma rez. + 4 coupler")

# --- GDS export (opsiyonel, zinciri kesmez) ---
import pandas as pd
if not hasattr(pd.DataFrame, "append"):
    def _append(self, other, ignore_index=False, **kw):
        if isinstance(other, dict):        other = pd.DataFrame([other])
        elif isinstance(other, pd.Series): other = other.to_frame().T
        elif isinstance(other, (list, tuple)):
            other = pd.concat([o if isinstance(o, pd.DataFrame) else pd.DataFrame([o]) for o in other])
        return pd.concat([self, other], ignore_index=ignore_index)
    pd.DataFrame.append = _append
try:
    gds = design.renderers.gds
    gds.options['short_segments_to_not_fillet'] = 'True'
    gds.export_to_gds("repetition_code_d3.gds")
    print("GDS kaydedildi: repetition_code_d3.gds")
except Exception as ex:
    print("GDS export atlandi (renderer purüzü, kritik degil):", type(ex).__name__, ex)
fig

## Faz 6 — Mantıksal qubit: devre-seviyesi hata düzeltme (Stim + PyMatching)

In [ ]:
# === FAZ 6 ===
import stim, pymatching, numpy as np, matplotlib.pyplot as plt

def logical_error(d, rounds, p, shots=30000):
    circ = stim.Circuit.generated("repetition_code:memory", distance=d, rounds=rounds,
        before_round_data_depolarization=p, before_measure_flip_probability=p)
    matcher = pymatching.Matching.from_detector_error_model(circ.detector_error_model(decompose_errors=True))
    dets, obs = circ.compile_detector_sampler().sample(shots, separate_observables=True)
    pred = matcher.decode_batch(dets)
    return np.mean(pred[:, 0] != obs[:, 0])

ps = np.logspace(-2.3, -0.7, 12)
plt.figure(figsize=(7.5, 5.5))
colors = {3:"#1f3b73", 5:"#1e8449", 7:"#c0392b"}
for d in [3, 5, 7]:
    pl = [logical_error(d, d, p) for p in ps]
    plt.loglog(ps, pl, 'o-', color=colors[d], lw=2, label=f"d={d}")
plt.loglog(ps, ps, 'k--', lw=1.3, label="mantiksal = fiziksel")

p_dev = 5e-3
pl_dev = logical_error(7, 7, p_dev, shots=200000)
plt.scatter([p_dev], [pl_dev], s=80, color="black", zorder=6)
plt.text(p_dev*1.2, pl_dev, f"  cihaz d=7\n  P_L={pl_dev:.1e}", fontsize=9, va="center")
plt.xlabel("fiziksel hata orani p"); plt.ylabel("mantiksal hata orani")
plt.title("Devre-seviyesi QEC: esik-alti hata bastirma (Stim)")
plt.legend(fontsize=9); plt.grid(alpha=0.3, which="both"); plt.tight_layout(); plt.show()
print(f"Cihaz (p={p_dev}, d=7): mantiksal hata = {pl_dev:.2e}")

---
### Özet
6 fazın tamamı: fiziksel transmon → parametre çıkarımı → kuantum dinamiği (kapılar, koherans) → iki-qubit kapı → tam çip → mantıksal qubit. Mesafe arttıkça (d=3→5→7) mantıksal hatanın üstel bastırılması, hata düzeltmenin çalıştığının kanıtıdır.

**Genişletme fikirleri:** yüzey kodu d=3 (17 qubit), akı-ayarlı transmon (SQUID), Ansys/Elmer ile gerçek EM çıkarımı, SQuADDS ile doğrulanmış parametreler.